# Nature Dependant People - Wetlands Data Processing

In [14]:
import pandas as pd
import geopandas as gpd
import json
import os
import numpy as np

In [15]:
gcs_path = "https://storage.googleapis.com/wetlands-gap-map/original_data"
analysis_data_folder = 'NDP'

In [16]:
locations = gpd.read_file(f'{gcs_path}/locations_simplified.geojson')
locations_sahel = locations[locations['type'] == 'global']
locations_countries = locations[locations['type'] == 'admin_region']
print(f'Total locations: {len(locations)}')
print(f'Total Sahel locations: {len(locations_sahel)}')
print(f'Total country locations: {len(locations_countries)}')


Total locations: 2068
Total Sahel locations: 1
Total country locations: 26


## Load NDP composite index data

In [17]:
file_path = f'{gcs_path}/{analysis_data_folder}/NDP_composite.shp'
ndp_data = gpd.read_file(file_path)
ndp_data['name_en'] = ndp_data['name_en'].str.replace('The Gambia', 'Gambia')
ndp_data = ndp_data.merge(locations_countries[['name', 'code']], left_on='name_en', right_on='name', how='left')
ndp_data.drop(columns=['name_x', 'name_y','fid', 'osm_id', 'name_en', 'boundary', 'admin_leve',
       'admin_cent', 'admin_ce_1', 'admin_ce_2', 'label_node', 'label_no_1',
       'label_no_2', 'layer', 'path', 'geometry'],
        inplace=True)
ndp_data.rename(columns={'GID_0': 'code'}, inplace=True)
ndp_data.dropna(subset=['code'], inplace=True)
ndp_data['location_id'] = ndp_data['code'].apply(lambda x: "adminregion_" + x.lower())
ndp_data.drop(columns=['code'], inplace=True)
ndp_data.columns = ndp_data.columns.str.lower().str.replace(' ', '_')
ndp_data.head(3)


,ndp_comp,ndp_water,location_id
0,0.297,0.17,adminregion_gmb
2,0.386,0.33,adminregion_mrt
3,0.134,0.19,adminregion_sen


In [18]:
# add sahel as average of all countries
sahel_ndp = {'location_id': 'global_sahel',
              'ndp_comp': round(ndp_data['ndp_comp'].mean(), 3),
              'ndp_water': round(ndp_data['ndp_water'].mean(), 3)}
ndp_data = pd.concat([ndp_data, pd.DataFrame([sahel_ndp])], ignore_index=True)
ndp_data['ndp_comp'] = round(ndp_data['ndp_comp'] * 100, 1)
ndp_data['ndp_water'] = round(ndp_data['ndp_water'] * 100, 1)
ndp_data.tail()

,ndp_comp,ndp_water,location_id
22,48.2,40.0,adminregion_cmr
23,0.0,0.0,adminregion_eri
24,73.5,38.0,adminregion_ssd
25,80.2,51.0,adminregion_tcd
26,47.4,37.3,global_sahel


## Export composite index data

In [19]:
indicator_data_list = []

for location in ndp_data['location_id'].unique():
    df_location = ndp_data[ndp_data['location_id'] == location]
    df_location.reset_index(drop=True, inplace=True)
    data_list = []
    for i in range(len(df_location)):
        data_list.append({
            "id":"ndp_composite_" + location + "_" + str(i),
            "label": "ndp_composite",
            "value": df_location.loc[i, 'ndp_comp'],
            "type":"",
            "group": "",
            "color":"#8377FF",
            "format": "percentage",
            "unit": "%"
        })
    data_json = json.dumps(data_list, indent=2)
    temp_dict = {"id": "ndp-composite-" + location,
                 "location": location,
                 "indicator": "ndp-composite",
                 "data": json.loads(data_json),
                 "locale": {"en": {"labels":{
                     "ndp_composite": "NDP composite index",
                    }
                    }},
                }
    indicator_data_list.append(temp_dict)

In [20]:
print(json.dumps(indicator_data_list, indent=2))

[
  {
    "id": "ndp-composite-adminregion_gmb",
    "location": "adminregion_gmb",
    "indicator": "ndp-composite",
    "data": [
      {
        "id": "ndp_composite_adminregion_gmb_0",
        "label": "ndp_composite",
        "value": 29.7,
        "type": "",
        "group": "",
        "color": "#8377FF",
        "format": "percentage",
        "unit": "%"
      }
    ],
    "locale": {
      "en": {
        "labels": {
          "ndp_composite": "NDP composite index"
        }
      }
    }
  },
  {
    "id": "ndp-composite-adminregion_mrt",
    "location": "adminregion_mrt",
    "indicator": "ndp-composite",
    "data": [
      {
        "id": "ndp_composite_adminregion_mrt_0",
        "label": "ndp_composite",
        "value": 38.6,
        "type": "",
        "group": "",
        "color": "#8377FF",
        "format": "percentage",
        "unit": "%"
      }
    ],
    "locale": {
      "en": {
        "labels": {
          "ndp_composite": "NDP composite index"
        }
 

### Save to JSON

In [21]:
seeding_json = json.load(open('../../app-initial-data/indicator-data.json'))
#remove all entries with id that starts with "ndp-composite-" and "ndp_composite_"
seeding_json = [entry for entry in seeding_json if not entry['id'].
                startswith('ndp-composite-') and not entry['id'].
                startswith('ndp_composite_')]

# Append the new data, if id exists update it
existing_ids = {entry['id'] for entry in seeding_json}
for new_entry in indicator_data_list:
    if new_entry['id'] in existing_ids:
        seeding_json = [entry if entry['id'] != new_entry['id'] else new_entry for entry in seeding_json]
    else:
        seeding_json.append(new_entry)  
with open('../../app-initial-data/indicator-data.json', 'w') as f:
    json.dump(seeding_json, f, indent=2)

## Export water index data

In [22]:
indicator_data_list = []

for location in ndp_data['location_id'].unique():
    df_location = ndp_data[ndp_data['location_id'] == location]
    df_location.reset_index(drop=True, inplace=True)
    data_list = []
    for i in range(len(df_location)):
        data_list.append({
            "id":"ndp_water_" + location + "_" + str(i),
            "label": "ndp_water",
            "value": df_location.loc[i, 'ndp_water'],
            "type":"",
            "group": "",
            "color":"#5AC4C6",
            "format": "percentage",
            "unit": "%"
        })
    data_json = json.dumps(data_list, indent=2)
    temp_dict = {"id": "ndp-water-" + location,
                 "location": location,
                 "indicator": "ndp-water",
                 "data": json.loads(data_json),
                 "locale": {"en": {"labels":{
                     "ndp_water": "NDP water index",
                    }
                    }},
                }
    indicator_data_list.append(temp_dict)

In [23]:
print(json.dumps(indicator_data_list, indent=2))

[
  {
    "id": "ndp-water-adminregion_gmb",
    "location": "adminregion_gmb",
    "indicator": "ndp-water",
    "data": [
      {
        "id": "ndp_water_adminregion_gmb_0",
        "label": "ndp_water",
        "value": 17.0,
        "type": "",
        "group": "",
        "color": "#5AC4C6",
        "format": "percentage",
        "unit": "%"
      }
    ],
    "locale": {
      "en": {
        "labels": {
          "ndp_water": "NDP water index"
        }
      }
    }
  },
  {
    "id": "ndp-water-adminregion_mrt",
    "location": "adminregion_mrt",
    "indicator": "ndp-water",
    "data": [
      {
        "id": "ndp_water_adminregion_mrt_0",
        "label": "ndp_water",
        "value": 33.0,
        "type": "",
        "group": "",
        "color": "#5AC4C6",
        "format": "percentage",
        "unit": "%"
      }
    ],
    "locale": {
      "en": {
        "labels": {
          "ndp_water": "NDP water index"
        }
      }
    }
  },
  {
    "id": "ndp-water-admin

In [24]:
seeding_json = json.load(open('../../app-initial-data/indicator-data.json'))
#remove all entries with id that starts with "ndp-water-" and "ndp_water_"
seeding_json = [entry for entry in seeding_json if not entry['id'].
                startswith('ndp-water-') and not entry['id'].
                startswith('ndp_water_')]

# Append the new data, if id exists update it
existing_ids = {entry['id'] for entry in seeding_json}
for new_entry in indicator_data_list:
    if new_entry['id'] in existing_ids:
        seeding_json = [entry if entry['id'] != new_entry['id'] else new_entry for entry in seeding_json]
    else:
        seeding_json.append(new_entry)  
with open('../../app-initial-data/indicator-data.json', 'w') as f:
    json.dump(seeding_json, f, indent=2)